In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, math, warnings
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/spikeparam')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/shared_helper_modules')

from ridge_regression_utils import (
    load_cell_data, load_hpf_lfp_windows, build_ridge_matrices,
    run_ridge_regression, apply_fdr, plot_ridge_results,
    WAVEFORM_LABELS,
)

warnings.filterwarnings('ignore', message='use_inf_as_na option is deprecated',
                        category=FutureWarning, module='seaborn')
warnings.filterwarnings('ignore', message=r'invalid value encountered in log10',
                        category=RuntimeWarning, module=r'.*specparam.*')

In [ ]:
# ── Cell configuration ────────────────────────────────────────────────────────
cell_num = 7

# Symmetric 50 ms windows immediately around the spike
PRE_WIN      = (-0.055, -0.005)  # pre-spike:  −55 → −5 ms
POST_WIN     = ( 0.005,  0.055)  # post-spike: +5 → +55 ms
BASELINE_WIN = (-0.20,  -0.10)   # baseline: −200 → −100 ms

# HPF detrending for LFP amp/std targets (0.1 Hz removes slow electrode drift)
HPF_CUTOFF = 0.1

# Ridge regression settings
ALPHAS          = np.logspace(-3, 3, 100)
N_PERM          = 1000
RNG_SEED        = 42
FORCE_RECOMPUTE = True   # True: rerun and overwrite pickle (needed after structure change)

In [ ]:
# ── 1. Load data ──────────────────────────────────────────────────────────────
df_reg, specparam_by_spike, lfp_windows_by_spike = load_cell_data(cell_num)

# HPF windows for the HPF version (0.1 Hz removes slow electrode drift)
hpf_lfp = load_hpf_lfp_windows(cell_num, hpf_cutoff=HPF_CUTOFF)[:len(specparam_by_spike)]

In [ ]:
# ── 2. Build matrices ─────────────────────────────────────────────────────────
# No HPF: raw LFP windows used for amp/std targets
(X_waveform, X_log_isi, X_both, waveform_labels,
 Y_raw, target_names, target_labels) = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
    hpf_lfp_by_spike=None,
)

# HPF 0.1 Hz: amp/std targets computed from high-pass filtered LFP trace
_, _, _, _, Y_hpf, _, _ = build_ridge_matrices(
    df_reg, specparam_by_spike, lfp_windows_by_spike,
    pre_win=PRE_WIN, post_win=POST_WIN, baseline_win=BASELINE_WIN,
    hpf_lfp_by_spike=hpf_lfp,
)

predictor_sets = {
    'Waveform only':      (X_waveform, waveform_labels),
    'Log ISI only':       (X_log_isi,  ['Log ISI']),
    'Waveform + Log ISI': (X_both,     waveform_labels + ['Log ISI']),
}

In [ ]:
# ── 3. Ridge regression — no HPF and HPF 0.1 Hz ──────────────────────────────
import os, sys
sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import SPE1_PICKLE_ROOT

RIDGE_DIR     = os.path.join(SPE1_PICKLE_ROOT, 'ridge_regression_pickles')
save_path_raw = os.path.join(RIDGE_DIR, f'c{cell_num}_ridge_results.pkl')
save_path_hpf = os.path.join(RIDGE_DIR, f'c{cell_num}_ridge_results_hpf.pkl')

results_raw = run_ridge_regression(
    Y_raw, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
    save_path=save_path_raw, force_recompute=FORCE_RECOMPUTE,
)
results_hpf = run_ridge_regression(
    Y_hpf, predictor_sets, target_names,
    n_perm=N_PERM, rng_seed=RNG_SEED, alphas=ALPHAS,
    save_path=save_path_hpf, force_recompute=FORCE_RECOMPUTE,
)

In [ ]:
# ── 4. FDR correction + save both versions ────────────────────────────────────
import pickle

for results, save_path in [(results_raw, save_path_raw), (results_hpf, save_path_hpf)]:
    results = apply_fdr(results, target_names, predictor_sets)
    with open(save_path, 'wb') as f:
        pickle.dump(results, f)
    print(f'Saved: {os.path.basename(save_path)}')

## 2. Build Feature and Target Matrices

**Predictors (X)** — three competing sets:
- *Waveform only*: 8 spike shape features
- *Log ISI only*: 1 feature
- *Waveform + Log ISI*: 9 features combined

**Targets (Y)** — 25 LFP scalars (5 groups × 5 features):

| Group | Formula | Scientific question |
|-------|---------|-------------------|
| Pre absolute | mean(pre window) | LFP state when cell fires |
| Pre−BL | mean(pre) − mean(baseline) | LFP ramp into spike |
| Post absolute | mean(post window) | LFP state after spike |
| Post−BL | mean(post) − mean(baseline) | Spike-triggered response from baseline |
| Δ post−pre | mean(post) − mean(pre) | Net spike-triggered change |

Windows: Baseline −200→−100 ms  |  Pre −55→−5 ms  |  Post +5→+55 ms